# 06 — Volatility Smiles and Surfaces

I use a simple synthetic surface so that I know the generating volatility, can recover it from prices, and can still ask whether the resulting price grid respects basic arbitrage restrictions.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))


In [2]:
from options_lab.volatility_surface import generate_synthetic_surface, recover_surface_implied_volatility, check_call_slice_arbitrage, check_calendar_total_variance

surface=generate_synthetic_surface(100.,.03,np.linspace(-.4,.4,17),[30/365,90/365,180/365,1.,2.])
surface.head()

,spot,rate,time_to_expiry,forward,strike,log_moneyness,implied_volatility,total_variance,call_price,put_price
0,100.0,0.03,0.082192,100.24688,67.197493,-0.40,0.27922,0.006408,32.967996,3.560265e-07
1,100.0,0.03,0.082192,100.24688,70.642782,-0.35,0.26847,0.005924,29.531195,3.551567e-06
2,100.0,0.03,0.082192,100.24688,74.264715,-0.30,0.25862,0.005497,25.918215,3.709953e-05
3,100.0,0.03,0.082192,100.24688,78.072348,-0.25,0.24967,0.005123,22.120302,3.800953e-04
4,100.0,0.03,0.082192,100.24688,82.075203,-0.20,0.24162,0.004798,18.130470,3.545628e-03


In [3]:
plt.figure(figsize=(8,5))
for maturity in sorted(surface.time_to_expiry.unique()):
    g=surface[surface.time_to_expiry==maturity]
    plt.plot(g.log_moneyness,100*g.implied_volatility,marker="o",label=f"T={maturity:.2f}")
plt.axvline(0,linestyle="--",linewidth=1)
plt.xlabel("Log-forward moneyness")
plt.ylabel("Implied volatility (%)")
plt.title("Synthetic Volatility Smiles")
plt.legend()
plt.show()

/tmp/ipykernel_7475/482371110.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [4]:
recovered=recover_surface_implied_volatility(surface,"call_price","call")
pd.Series({"max_iv_recovery_error":recovered.iv_absolute_error.max(),
           "all_strike_checks_pass":check_call_slice_arbitrage(surface).all_checks_passed.all(),
           "all_calendar_checks_pass":check_calendar_total_variance(surface).calendar_check_passed.all()})

max_iv_recovery_error        0.0
all_strike_checks_pass      True
all_calendar_checks_pass    True
dtype: object

**What I took from it.** A smooth surface is not automatically a valid one. Price bounds, strike monotonicity, convexity, and calendar total variance still need to be checked.